# 里程碑 1：双均线交叉端到端跑通

验证链路：`akshare 取数 → 规整落盘 → 算指标 → 生成信号 → 回测 → 统计`。

标的：`000001`（平安银行），日线，后复权。

**为什么用后复权**：前复权的历史价格会在每次分红后被追溯性重写，同一段历史今天跑和
下个月跑结果不同，回测不可复现。后复权数值稳定且不含未来函数。

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
import vectorbt as vbt

import ashare
from ashare.config import DEFAULT_ADJUST, DEFAULT_FEES
from ashare.data import get_prices, trade_calendar
from ashare.strategy import MAParams, drop_limit_signals, limit_pct, ma_cross_signals
from ashare.backtest import run_signals

# 下面两行都只为控制 notebook 体积，不影响回测结果。
#
# vectorbt 的 .plot() 默认返回 ipywidgets 控件（use_widgets=True），控件状态会连同
# 整份行情数据序列化进 notebook 的 metadata.widgets，实测 10.3MB —— 且没有任何
# cell 输出引用它，纯属死重。关掉后 .plot() 返回普通 plotly Figure，缩放/悬停照常，
# 只是少了控件面板。
vbt.settings["plotting"]["use_widgets"] = False
# plotly.js 交给 CDN，不内联进每张图（默认渲染器每张图要塞 ~3.5MB 的库）。
pio.renderers.default = "notebook_connected"

print("vectorbt", vbt.__version__)
print("pandas  ", pd.__version__)
print("numpy   ", np.__version__)
print("ashare  ", ashare.__version__)

vectorbt 1.1.0
pandas   3.0.6
numpy    2.4.6
ashare   0.1.0


## 1. 取数（首次走网络，之后走 parquet 缓存）

In [2]:
SYMBOL = "000001"
START, END = "20180101", "20260919"

df = get_prices(SYMBOL, START, END, adjust=DEFAULT_ADJUST)
print(f"{len(df)} 行  {df.index.min().date()} ~ {df.index.max().date()}")
df.tail()

2116 行  2018-01-02 ~ 2026-09-18


,Open,High,Low,Close,Volume,Amount,PctChg,Turnover
Date,,,,,,,,
2026-09-14,1768.02,1793.64,1766.51,1786.10,74124802.0,8.760710e+08,0.936977,0.381975
2026-09-15,1781.58,1790.63,1774.05,1781.58,75525146.0,8.933829e+08,-0.253065,0.389191
2026-09-16,1778.57,1784.60,1743.90,1763.49,94962632.0,1.106653e+09,-1.015391,0.489355
2026-09-17,1760.48,1769.52,1743.90,1749.93,69192535.0,8.061531e+08,-0.768930,0.356558
2026-09-18,1746.91,1781.58,1742.39,1763.49,85303779.0,9.999181e+08,0.774888,0.439581


In [3]:
# 数据质量断言：索引单调、OHLC 无缺失
assert isinstance(df.index, pd.DatetimeIndex)
assert df.index.is_monotonic_increasing
assert not df.index.has_duplicates
assert df[["Open", "High", "Low", "Close"]].notna().all().all()
_cal = trade_calendar()
print("缓存区间内的交易日数:", _cal.to_series().between(df.index.min(), df.index.max()).sum())
print("实际行数            :", len(df))
print("差值即停牌天数      :", _cal.to_series().between(df.index.min(), df.index.max()).sum() - len(df))

缓存区间内的交易日数: 2116
实际行数            : 2116
差值即停牌天数      : 0


In [4]:
# 缓存命中验证：第二次调用应显著快于首次
import time

t0 = time.perf_counter()
df2 = get_prices(SYMBOL, START, END, adjust=DEFAULT_ADJUST)
elapsed = time.perf_counter() - t0

pd.testing.assert_frame_equal(df, df2)
print(f"缓存命中，耗时 {elapsed * 1000:.1f} ms")

缓存命中，耗时 2.8 ms


## 2. 价格走势

In [5]:
close = df["Close"]
close.vbt.plot(trace_kwargs=dict(name=f"{SYMBOL} 后复权收盘"), width=1000, height=400).show()

## 3. 信号

双均线交叉，并剔除落在涨跌停价上的入场信号 —— 那些价位实际挂不出单，不回剔会让
结果偏乐观。注意剔除的只是「信号当天已封板」，次日跳空封板仍然无法建模。

In [6]:
params = MAParams(fast=10, slow=50)
entries, exits = ma_cross_signals(close, params)

entries_nofilter = entries.copy()
entries = drop_limit_signals(entries, df["PctChg"], limit_pct(SYMBOL))

print(f"参数: fast={params.fast} slow={params.slow}")
print(f"原始入场信号 {entries_nofilter.sum()} 个，剔除涨跌停后 {entries.sum()} 个")
print(f"出场信号 {exits.sum()} 个")

参数: fast=10 slow=50
原始入场信号 28 个，剔除涨跌停后 27 个
出场信号 27 个


## 4. 回测

In [7]:
pf = run_signals(close, entries, exits)
print(f"初始资金 {pf.init_cash:,.0f}   费率 {DEFAULT_FEES}（对称，近似双边成本）")
pf.stats()

初始资金 100,000   费率 0.0008（对称，近似双边成本）


Start                             2018-01-02 00:00:00
End                               2026-09-18 00:00:00
Period                             2116 days 00:00:00
Start Value                                  100000.0
End Value                               125283.688944
Total Return [%]                            25.283689
Benchmark Return [%]                        11.603402
Max Gross Exposure [%]                          100.0
Total Fees Paid                           5570.909471
Max Drawdown [%]                            41.511011
Max Drawdown Duration              1363 days 00:00:00
Total Trades                                       27
Total Closed Trades                                26
Total Open Trades                                   1
Open Trade PnL                            9758.481507
Win Rate [%]                                34.615385
Best Trade [%]                              47.716038
Worst Trade [%]                            -14.843345
Avg Winning Trade [%]       

In [8]:
assert pf.stats()["Total Trades"] > 0, "没有产生任何交易，检查信号或数据"
assert np.isfinite(pf.total_return()), "总收益不是有限值"

trades = pf.trades.records_readable
assert (trades["Exit Timestamp"] >= trades["Entry Timestamp"]).all(), "存在平仓早于开仓的记录"
print(f"{len(trades)} 笔交易，断言全部通过")
trades.head(10)

27 笔交易，断言全部通过


,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
0,0,0,85.035457,2018-08-24,1175.04,79.936051,2018-11-27,1192.62,81.131990,1333.855301,0.013349,Long,Closed,0
1,1,0,83.586096,2019-01-21,1211.36,81.002282,2019-05-14,1463.24,97.845215,20874.818315,0.206165,Long,Closed,1
2,2,0,77.611105,2019-06-25,1573.37,97.688788,2019-11-26,1849.90,114.858227,21249.251948,0.174016,Long,Closed,2
3,3,0,73.801538,2019-12-24,1942.28,114.674601,2020-02-03,1656.86,97.823053,-21276.932591,-0.148433,Long,Closed,3
4,4,0,73.894951,2020-05-08,1652.12,97.666661,2020-06-19,1541.78,91.143806,-8342.379351,-0.068333,Long,Closed,4
5,5,0,65.670730,2020-08-12,1732.09,90.998092,2021-03-11,2562.01,134.599255,54275.855280,0.477160,Long,Closed,5
6,6,0,60.792746,2021-04-27,2763.16,134.384068,2021-06-24,2801.88,136.267184,2083.243884,0.012402,Long,Closed,6
7,7,0,75.599089,2021-09-17,2249.52,136.049331,2021-09-23,2146.33,129.808475,-8066.927835,-0.047435,Long,Closed,7
8,8,0,69.178652,2021-10-18,2341.78,129.600947,2021-11-08,2114.77,117.037551,-15950.884320,-0.098462,Long,Closed,8
9,9,0,75.575784,2022-04-12,1932.67,116.850440,2022-05-09,1766.35,106.794629,-12793.409440,-0.087588,Long,Closed,9


In [9]:
pf.plot().show()

## 5. 手工核对一笔交易

抽查第一笔，确认成交价确实取自信号次日、手续费等于成交额乘以费率。

In [10]:
t = trades.iloc[0]
print(t[["Entry Timestamp", "Avg Entry Price", "Exit Timestamp",
         "Avg Exit Price", "PnL", "Entry Fees", "Exit Fees"]].to_string())

# 手续费核对：成交额 × 费率
notional = t["Size"] * t["Avg Entry Price"]
expected_fee = notional * DEFAULT_FEES
print(f"\n买入成交额 {notional:,.2f} × 费率 {DEFAULT_FEES} = {expected_fee:,.2f}")
print(f"记录中的 Entry Fees                    = {t['Entry Fees']:,.2f}")
assert abs(expected_fee - t["Entry Fees"]) < 1.0, "手续费与 成交额×费率 不符"
print("手续费核对通过")

# 成交价应取自信号次日
sig_dates = entries_nofilter[entries_nofilter].index
print(f"\n首个入场信号日 {sig_dates[0].date()}，交易实际入场日 {t['Entry Timestamp'].date()}")
print(f"入场日收盘价 {close.loc[t['Entry Timestamp']]:,.2f}，交易记录均价 {t['Avg Entry Price']:,.2f}")

Entry Timestamp    2018-08-24 00:00:00
Avg Entry Price                1175.04
Exit Timestamp     2018-11-27 00:00:00
Avg Exit Price                 1192.62
PnL                        1333.855301
Entry Fees                   79.936051
Exit Fees                     81.13199

买入成交额 99,920.06 × 费率 0.0008 = 79.94
记录中的 Entry Fees                    = 79.94
手续费核对通过

首个入场信号日 2018-08-24，交易实际入场日 2018-08-24
入场日收盘价 1,175.04，交易记录均价 1,175.04


## 6. 已知偏差（解读结果时必须考虑）

1. **涨跌停**：开源版 vectorbt 无限价单，只能按信号价成交。信号日封板的单已剔除，但
   次日跳空封板仍无法成交 —— 结果仍偏乐观。
2. **费率对称**：真实成本是佣金（约 0.0003，有 5 元起收）+ 印花税 0.0005（仅卖出）
   + 过户费 0.00001。`fees=0.0008` 对称征收，大致匹配双边总额但高估了买入腿。
   小资金、高换手策略受此影响更大。
3. **无滑点**：成交价直接取收盘价，未建模冲击成本和买卖价差。
4. **T+1**：信号用收盘价、次日执行，已天然满足 T+1，无需额外处理。但若改成当日
   开盘执行就会违反 T+1，注意不要随意调整 `shift` 参数。
5. **停牌**：停牌日在缓存中不存在（非 NaN 填充），vectorbt 不会在停牌 bar 上成交，
   但持仓市值会跨停牌期连续计算。
6. **单只单参数**：本 notebook 只验证链路，未做样本内外分割。换参数直到好看就是
   过拟合，参数寻优必须配合 `vbt.rolling_split()`。
7. **数据来源不唯一**：`fetch_hist` 在 eastmoney → sina → tencent 之间按序回退，
   实测东财会限流。三家口径已在本层统一（涨跌幅自算、换手率统一为百分数），但
   成交额/换手率的原始定义仍有细微差别。缓存元信息里的 `source` 字段记录了每个
   区间实际来自哪家，跨源拼接的长区间要注意这一点 —— 本次运行的 source 见上文
   取数单元格输出。